# Fase 5 - Avaliacao e IA Responsavel

Explicabilidade (traducao NL) e Monitoramento (estabilidade temporal).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import (
    ANOS_OCORRENCIA,
    COLUNAS_CONTEXTO,
    DATA_DIR,
    DADOS_DIR,
    FIGURAS_DIR,
    FILTRO_UF,
    MIN_SUPPORT,
    PROCESSED_DIR,
    TABELAS_DIR,
)

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_style("whitegrid")
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)
from src.evaluation import disclaimer, exportar_top_regras_latex, resumo_qualidade_regras, traduzir_regras

## 5.1 Regras

In [2]:
rules = pd.read_csv(DADOS_DIR / "regras_contexto_fatal.csv")
for col in ("lift", "confidence", "support"):
    rules[col] = pd.to_numeric(rules[col])
print(resumo_qualidade_regras(rules))

{'n_regras': 76, 'lift_max': np.float64(5.4359), 'lift_mediano': np.float64(1.8086)}


## 5.2 Explicabilidade

In [3]:
for i, txt in enumerate(rules["explicacao_natural"].head(10), 1):
    print(f"{i:2d}. {txt}")

 1. Quando fase=dia Plena Noite E tipo=acidente Atropelamento de Pedestre E uso=solo Não, entao desfecho fatal. [Suporte: 0.5% | Confianca: 42.6% | Lift: 5.44]
 2. Quando causa=acidente Transitar na contramão E tipo=acidente Colisão frontal E uso=solo Não, entao desfecho fatal. [Suporte: 0.7% | Confianca: 42.3% | Lift: 5.40]
 3. Quando causa=acidente Transitar na contramão E tipo=acidente Colisão frontal E tipo=pista Simples, entao desfecho fatal. [Suporte: 0.8% | Confianca: 38.9% | Lift: 4.97]
 4. Quando condicao=metereologica Céu Claro E tipo=acidente Colisão frontal E uso=solo Não, entao desfecho fatal. [Suporte: 1.1% | Confianca: 37.7% | Lift: 4.82]
 5. Quando fase=dia Plena Noite E tipo=acidente Colisão frontal E uso=solo Não, entao desfecho fatal. [Suporte: 0.7% | Confianca: 37.6% | Lift: 4.80]
 6. Quando tipo=acidente Colisão frontal E tracado=via Reta E uso=solo Não, entao desfecho fatal. [Suporte: 0.9% | Confianca: 37.5% | Lift: 4.79]
 7. Quando fim=de semana Sim E tipo=aciden

## 5.3 Estabilidade temporal

In [4]:
est = pd.read_csv(TABELAS_DIR / "regras_estabilidade_temporal.csv")
estaveis = est[est["status"] == "estavel"]
print(f"Estaveis: {len(estaveis)} | Transitorias: {(est['status']=='transitoria').sum()}")
display(estaveis[["regra", "n_anos_presente", "lift_medio", "confianca_media"]].head(10))

Estaveis: 87 | Transitorias: 57


,regra,n_anos_presente,lift_medio,confianca_media
0,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_At...",2,5.6527,0.4371
1,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,5.5566,0.4393
2,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,5.0138,0.3959
3,"ctx_fase_dia_Plena_Noite, ctx_tipo_acidente_Co...",4,4.9423,0.3904
4,"ctx_fim_de_semana_Sim, ctx_tipo_acidente_Colis...",3,4.9066,0.3895
5,"ctx_tipo_acidente_Colisão_frontal, ctx_tracado...",3,4.8900,0.3883
6,"ctx_condicao_metereologica_Céu_Claro, ctx_tipo...",4,4.8476,0.3825
7,"ctx_condicao_metereologica_Céu_Claro, ctx_fim_...",2,4.7893,0.3744
8,"ctx_causa_acidente_Transitar_na_contramão, ctx...",4,4.7833,0.3777
9,"ctx_condicao_metereologica_Céu_Claro, ctx_fase...",3,4.6937,0.3728


## 5.4 Qualidade

In [5]:
rules["faixa_lift"] = pd.cut(rules["lift"], bins=[1.05, 1.5, 2, 3, 10], labels=["1-1.5", "1.5-2", "2-3", "3+"])
print(rules["faixa_lift"].value_counts().sort_index())
print(f"Regras lift >= 3: {(rules['lift'] >= 3).sum()}")

faixa_lift
1-1.5    22
1.5-2    19
2-3       0
3+       35
Name: count, dtype: int64
Regras lift >= 3: 35


## 5.5 Limitacoes

In [6]:
print(disclaimer())

AVISO: Regras representam coocorrencias estatisticas. Associacao nao implica causalidade.


## 5.6 Export LaTeX

In [7]:
if (PROCESSED_DIR / "rules_fatal.pkl").exists():
    rules_pk = pd.read_pickle(PROCESSED_DIR / "rules_fatal.pkl")
    exportar_top_regras_latex(rules_pk, TABELAS_DIR / "regras_top15_latex.tex")
    traduzir_regras(rules_pk).head(15).to_csv(TABELAS_DIR / "top15_regras_traduzidas.csv", index=False)
    print("[OK] LaTeX e CSV exportados")

[OK] LaTeX e CSV exportados
